Import packages

In [36]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [37]:
from pathlib import Path
import plotly
import plotly.express as px
import scipy.stats
os.environ["OMP_NUM_THREADS"] = '1' # because of a data leak of KMeans on windows
import fba_comparison_stats as cmp
from efflux_method import *
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

Import the models

How to make each models, how to put information in it? Efflux?

In [38]:
M_xanthus = read_sbml_model("../M_xanthus_model_V4.xml")

In [39]:
list_of_genes = []
for i in M_xanthus.genes:
    list_of_genes.append(i.id)

In [40]:
# Getting fluxes with constrains
dico = {14: "Alone", 2: "P_E_coli", 5: "P_B_subtilis", 8: "P_Caulobacter", 11: "P_Yeast"}
for i in [14, 2, 5, 8, 11]:
    M_xanthus = read_sbml_model("../M_xanthus_model_V4.xml")
    DictAP = read_csv_data(
        "/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys_iMAT.csv",
        id_gene=1,
        id_val=i,
        list_of_genes = list_of_genes,
        head=True,
        quantile=0.95,
    )
    Eflux(M_xanthus, DictAP, const=100, default_exp_val=1, ignore_human=True)
    M_xanthus.reactions.EX_glc_D_e.bounds = (0, 1000)
    model_c = M_xanthus.copy()
    write_sbml_model(
        model_c,
        "/home/mickael/github/M_xanthus-E_coli-Predation/fba_comparer-main/Models_Preda/M_xanthus_V4_efflux_"
        + dico[i]
        + ".xml",
    )
    print(dico[i] + " Done!")

Alone Done!
P_E_coli Done!
P_B_subtilis Done!
P_Caulobacter Done!
P_Yeast Done!


In [41]:
M_xanthus_alone = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_Alone.xml")
solution_alone = M_xanthus_alone.optimize()

In [42]:
M_xanthus_predation = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_P_E_coli.xml")
solution_predation_E = M_xanthus_predation.optimize()

In [43]:
M_xanthus_predation = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_P_B_subtilis.xml")
solution_predation_B = M_xanthus_predation.optimize()

In [44]:
M_xanthus_predation = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_P_Caulobacter.xml")
solution_predation_C = M_xanthus_predation.optimize()

In [45]:
M_xanthus_predation = read_sbml_model("Models_Preda/M_xanthus_V4_efflux_P_Yeast.xml")
solution_predation_Y = M_xanthus_predation.optimize()

In [46]:
print(f"Alone:\t{solution_alone.objective_value}")
print(f"Predation E.coli :\t{solution_predation_E.objective_value}")
print(f"Predation B_subtilis:\t{solution_predation_B.objective_value}")
print(f"Predation Caulobacter:\t{solution_predation_C.objective_value}")
print(f"Predation Yeast:\t{solution_predation_Y.objective_value}")

Alone:	0.204452758536134
Predation E.coli :	0.10165517381424201
Predation B_subtilis:	0.15158979104534828
Predation Caulobacter:	0.16752521568295686
Predation Yeast:	0.11658426982424433


## **FBA Comparer**
**Create combined dataframe and preprocess data**

Filter all reactions where fluxes are zero:

In [47]:
solutions = [solution_alone, solution_predation_E, solution_predation_B, solution_predation_C, solution_predation_Y] 
conditions = ["Alone", "Predation E.coli", "Predation B. subtilis", "Predation Caulobacter", "Predation Yeast"]
obj_values = [solution_alone.objective_value, solution_predation_E.objective_value, solution_predation_B.objective_value, solution_predation_C.objective_value, solution_predation_Y.objective_value]

mxanthus = cmp.build_dataframe(M_xanthus_alone, solutions, conditions)
mxanthus_filtered = cmp.filter_dataframe(mxanthus, conditions, rounding = True)
print(f"Number of reactions in the models: {len(mxanthus)}")
print(f"Number of nonzero reactions in the models: {len(mxanthus_filtered)}")

Number of reactions in the models: 1339
Number of nonzero reactions in the models: 446


normalize data:

In [48]:
mxanthus_normalized = cmp.normalize_dataframe_cols(mxanthus_filtered, conditions, obj_values) # biomass normalization
mxanthus_normalized_div_by_max = cmp.normalize_dataframe_rows(mxanthus_normalized, conditions, "div_by_max") # normalization reactions
# note: there is no difference between the order of div_by_max normalization and filtering for changing reactions
mxanthus_normalized_div_by_max_changing = mxanthus_normalized_div_by_max[mxanthus_normalized_div_by_max['Std_dev'] >= 0.01] # keep only reactions that change
print(f"Number of normalized changing reactions: {len(mxanthus_normalized_div_by_max_changing)}")

Number of normalized changing reactions: 197


## Identify reactions with most/least variation

Most affected / change reaction

In [49]:
most_variable_bar_plot = cmp.bar_plot_flux_variation(mxanthus_normalized, conditions, 0, 20)
most_variable_bar_plot.show()

In [50]:
sorted_flux = mxanthus_normalized[["ID", "Name", "Std_dev"]].sort_values(by=['Std_dev'], ascending=False) 
sorted_flux["Kegg"] = [M_xanthus_alone.reactions.get_by_id(id).__dict__["_annotation"].get("kegg.reaction") for id in sorted_flux["ID"]]
sorted_flux[0:20]

,ID,Name,Std_dev,Kegg
842,rxn01507_c,2'-Deoxyadenosine 5'-monophosphate phosphohydr...,5.114130,R02088
583,rxn01508_c,ATP:deoxyadenosine 5'-phosphotransferase [c],5.114130,R02089
909,rxn00119_c,ATP:UMP phosphotransferase [c],3.522258,R00158
352,rxn00463_c,Uridine triphosphate pyrophosphohydrolase [c],3.365245,R00662
1067,rxn00117_c,ATP:UDP phosphotransferase [c],3.365245,R00156
1261,EX_pi_e,Exchange for Phosphate [e],3.202937,None
672,rxn05312_c,Inorganic phosphate transporter [c],3.202937,None
716,rxn01366_c,Uridine:phosphate alpha-D-ribosyltransferase [c],3.124874,R01876
910,rxn00709_c,ATP:uridine 5'-phosphotransferase [c],3.124874,R00964
499,rxn00711_c,UMP:diphosphate phospho-alpha-D-ribosyltransfe...,2.818808,R00966


Pathway analysis

In [51]:
from bioservices import KEGG

k = KEGG()

for rxn in M_xanthus.reactions:
    if "kegg.reaction" in rxn.annotation:
        kegg_id = rxn.annotation["kegg.reaction"]
        try:
            data = k.get(kegg_id)
            parsed = k.parse(data)
            if "PATHWAY" in parsed:
                rxn.subsystem = list(parsed["PATHWAY"].values())[0]
        except:
            continue

WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02371)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R01549)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02097)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R02292)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R00962)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https://rest.kegg.jp/get/R00967)
WARNING [bioservices.KEGG:1210]:  Could not parse the entry correctly.
WARNING [bioservices.KEGG:535]:  HTTP 404 Not Found (https

In [52]:
pathway_dict_M = {}
for i in M_xanthus.reactions:
    if i.subsystem in pathway_dict_M:
        pathway_dict_M[i.subsystem].append(i.id)
    else:
        pathway_dict_M[i.subsystem] = [i.id]

In [53]:
pw_list = cmp.pathway_variability(mxanthus_normalized, pathway_dict_M, ascending=True, num=20)

Pyrimidine metabolism		 -> 0.93 (contains 25 reactions)
Purine metabolism		 -> 0.89 (contains 35 reactions)
Glycerophospholipid metabolism		 -> 0.57 (contains 1 reactions)
Glycolysis / Gluconeogenesis		 -> 0.51 (contains 6 reactions)
Arginine biosynthesis		 -> 0.48 (contains 7 reactions)
Metabolic pathways		 -> 0.37 (contains 1 reactions)
Citrate cycle (TCA cycle)		 -> 0.34 (contains 8 reactions)
Alanine, aspartate and glutamate metabolism		 -> 0.27 (contains 4 reactions)
Glyoxylate and dicarboxylate metabolism		 -> 0.18 (contains 3 reactions)
Pyruvate metabolism		 -> 0.17 (contains 1 reactions)
		 -> 0.17 (contains 152 reactions)
Valine, leucine and isoleucine degradation		 -> 0.13 (contains 4 reactions)
Phenylalanine metabolism		 -> 0.13 (contains 2 reactions)
Glycine, serine and threonine metabolism		 -> 0.10 (contains 9 reactions)
One carbon pool by folate		 -> 0.09 (contains 5 reactions)
Fatty acid elongation		 -> 0.09 (contains 6 reactions)
Fatty acid degradation		 -> 0.09 (conta

In [54]:
for i in M_xanthus.reactions._dict:
    if M_xanthus.reactions.get_by_id(i).subsystem == "Pyrimidine metabolism":
        print(i)

rxn00364_c
rxn01673_c
rxn00107_c
rxn01219_c
rxn01218_c
rxn00710_c
rxn00717_c
rxn01217_c
rxn01362_c
rxn01465_c
rxn00369_c
rxn00708_c
rxn00463_c
rxn01146_c
rxn01518_c
rxn00363_c
rxn00776_c
rxn01145_c
rxn00711_c
rxn01678_c
rxn00412_c
rxn01025_c
rxn01018_c
rxn00714_c
rxn00410_c
rxn00365_c
rxn01366_c
rxn01143_c
rxn01517_c
rxn01800_c
rxn01512_c
rxn00713_c
rxn01648_c
rxn01368_c
rxn06076_c
rxn01521_c
rxn00119_c
rxn00709_c
rxn00409_c
rxn01799_c
rxn01519_c
rxn01513_c
rxn06075_c
rxn00117_c
rxn01520_c
rxn00367_c
rxn16149_c
rxn00797_c


Flux correlation network

In [55]:
corr_matrix = cmp.flux_coupling_matrix(mxanthus_normalized_div_by_max, conditions, remove_unchanging=0.01, abs_values=True)

249 reaction were removed, because their Std_dev is lower than 0.01


In [56]:
nw = cmp.flux_coupling_network(corr_matrix, pc=0.8)
fig = cmp.visualize_interactive_network(nw, mxanthus_normalized_div_by_max, conditions)
fig.show()

Created Flux Coupling network with pc threshold 0.8.
Number of nodes: 197
Number of edges: 2694
Number of connected components: 9


In [57]:
nw2 = nw
cmp.color_network_by_communities(nw2, 8, include_weights=True) # takes edge weights into account!

In [58]:
fig = cmp.visualize_interactive_network(nw2, mxanthus_normalized_div_by_max, conditions)
fig.show()

Community map

later on

In [59]:
comm_map = cmp.get_community_list(nw)
color = cmp.get_color_map(comm_map)

In [60]:
fig = cmp.community_bar_plots(mxanthus_normalized_div_by_max, conditions, comm_map)
fig.show()

In [61]:
fig = cmp.community_box_plots(mxanthus_normalized_div_by_max, conditions, comm_map)
fig.show()